In [ ]:
#drop and recreate db
import os
os.remove("crystal_xrd.db")  # or the one you misspelled

In [1]:
import sqlite3

#create and connect to the database
#how to delete : crystal_crd.db i misspelled 
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

In [1]:
import sqlite3
con = sqlite3.connect("crystal_xrd.db")
con.execute("PRAGMA journal_mode=DELETE;")  # returns ('delete',)
con.execute("PRAGMA wal_checkpoint(TRUNCATE);")
con.close()


## open DB Connection

In [8]:
#open connection
import sqlite3
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

## Close connection

In [1]:
import sqlite3
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()
conn.close()

In [1]:
import os, sqlite3
p = os.path.abspath("crystal_xrd.db")
print(p)
sqlite3.connect(p).close()


/home/UserD/OrgA/crystal_database/crystal_xrd.db


## Creating final structures table

In [1]:
import sqlite3

#create and connect to the database
#how to delete : crystal_crd.db i misspelled 
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()
cursor.execute("DROP TABLE Structures;")
#create structures table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Structures (
structure_id TEXT PRIMARY KEY,
space_group INTEGER,
source TEXT,
source_id TEXT,
generation_commit TEXT,
extinction_class TEXT,
extinction_group INTEGER,
crystal_system TEXT,
composition TEXT,
lattice_a REAL,
lattice_b REAL,
lattice_c REAL,
alpha REAL,
beta REAL,
gamma REAL,
wyckoff_positions TEXT,
atom_sites_json TEXT,
creation_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
#commit and close
conn.commit()
conn.close()

## Create Patterns table

In [ ]:
import sqlite3

#create and connect to the database
#how to delete : crystal_crd.db i misspelled 
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

#create patterns table
cursor.execute("""
CREATE TABLE IF NOT EXISTS Patterns(
pattern_id TEXT PRIMARY KEY,
structure_id TEXT,
radiation_type TEXT,
wavelength REAL,
twotheta_min REAL,
twotheta_max REAL,
step_size REAL,
resolution_u REAL,
resolution_v REAL, 
resolution_w REAL,
background_type TEXT,
generator_method TEXT,
generator_commit TEXT,
normalization_type TEXT,
axis_type TEXT,
pattern_format TEXT,
x_axis_file TEXT,
intensity_file TEXT,
has_structure_link BOOLEAN DEFAULT 1,
comments TEXT,
generation_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
FOREIGN KEY (structure_id) REFERENCES Structures(structure_id)

);

""")

#commit and close
conn.commit()
conn.close()

In [3]:
import sqlite3

#create and connect to the database
#how to delete : crystal_crd.db i misspelled 
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

#create patterns table
cursor.execute("""
select count(*) FROM Structures where source != "PyXtal"

""")

#commit and close
conn.commit()
conn.close()

In [5]:
import sqlite3

conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE Patterns_old;")
# Step 1: Rename the existing Patterns table
cursor.execute("ALTER TABLE Patterns RENAME TO Patterns_old;")

# Step 2: Create the new Patterns table without `x_axis_file`
cursor.execute("""
CREATE TABLE Patterns (
    pattern_id TEXT PRIMARY KEY,
    structure_id TEXT,
    radiation_type TEXT,
    wavelength REAL,
    twotheta_min REAL,
    twotheta_max REAL,
    step_size REAL,
    resolution_u REAL,
    resolution_v REAL, 
    resolution_w REAL,
    background_type TEXT,
    generator_method TEXT,
    generator_commit TEXT,
    normalization_type TEXT,
    axis_type TEXT,
    pattern_format TEXT,
    intensity_file TEXT,
    has_structure_link BOOLEAN DEFAULT 1,
    comments TEXT,
    generation_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (structure_id) REFERENCES Structures(structure_id)
);
""")

# Step 3: Copy data (excluding x_axis_file)
cursor.execute("""
INSERT INTO Patterns (
    pattern_id,
    structure_id,
    radiation_type,
    wavelength,
    twotheta_min,
    twotheta_max,
    step_size,
    resolution_u,
    resolution_v,
    resolution_w,
    background_type,
    generator_method,
    generator_commit,
    normalization_type,
    axis_type,
    pattern_format,
    intensity_file,
    has_structure_link,
    comments,
    generation_timestamp
)
SELECT
    pattern_id,
    structure_id,
    radiation_type,
    wavelength,
    twotheta_min,
    twotheta_max,
    step_size,
    resolution_u,
    resolution_v,
    resolution_w,
    background_type,
    generator_method,
    generator_commit,
    normalization_type,
    axis_type,
    pattern_format,
    intensity_file,
    has_structure_link,
    comments,
    generation_timestamp
FROM Patterns_old;
""")

# Step 4: Drop old table
cursor.execute("DROP TABLE Patterns_old;")

# Done
conn.commit()
conn.close()

print("✅ Column `x_axis_file` removed and table updated successfully.")


✅ Column `x_axis_file` removed and table updated successfully.


## Recreating Patterns
----
revisiting the patterns table to create it with updates;
- Remove: x_axis_file and intensity_file (if not using .npy)
- Add: intensity_blob BLOB and optionally twotheta_blob BLOB

----
Then step two is to Load HDF5 into The newly created patterns table

- Read each pattern from the HDF5 file (like 2K_reflection.hdf5)
- Extract the space_group, twotheta, and intensity
- Store twotheta and intensity as binary blobs
- Insert the full row into the new Patterns table

In [11]:
import sqlite3

# Connect to your database
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

# Drop the old Patterns table (if it exists)
#cursor.execute("DROP TABLE IF EXISTS Patterns")

# Recreate the new Patterns table with BLOB fields
cursor.execute("""
CREATE TABLE IF NOT EXISTS Patterns2 (
    pattern_id TEXT PRIMARY KEY,
    structure_id TEXT,
    space_group INTEGER,
    radiation_type TEXT,
    wavelength REAL,
    twotheta_min REAL,
    twotheta_max REAL,
    step_size REAL,
    resolution_u REAL,
    resolution_v REAL, 
    resolution_w REAL,
    background_type TEXT,
    generator_method TEXT,
    generator_commit TEXT,
    normalization_type TEXT,
    axis_type TEXT,
    pattern_format TEXT,
    intensity_blob BLOB,
    twotheta_blob BLOB,
    has_structure_link BOOLEAN DEFAULT 1,
    comments TEXT,
    generation_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (structure_id) REFERENCES Structures(structure_id)
);
""")

# Commit and close
conn.commit()
conn.close()

print("✅ Dropped old Patterns table and created the new one with BLOB fields.")


✅ Dropped old Patterns table and created the new one with BLOB fields.


the script that wloads all reflection patterns from an HDF5 file (like 2K_reflection.hdf5) and insert them into our newly updated Patterns table with twotheta and intensity stored as BLOBs (binary data).

In [15]:
import sqlite3
import uuid
import h5py
import numpy as np

# Path to the HDF5 reflection file
HDF5_FILE = "2K_reflection.hdf5"
DB_FILE = "crystal_xrd.db"

# Static metadata from Lizzy's config
STATIC_METADATA = {
    "radiation_type": "X-ray",
    "wavelength": 1.54030,
    "twotheta_min": 10.0,
    "twotheta_max": 110.0,
    "step_size": 0.05,
    "resolution_u": 0.025,
    "resolution_v": 0.025,
    "resolution_w": 0.025,
    "background_type": "None",
    "generator_method": "Reflection",
    "generator_commit": "unknown",
    "normalization_type": "Max intensity = 1000",
    "axis_type": "2theta",
    "pattern_format": "hdf5",
    "has_structure_link": False,
    "comments": "Bulk-imported from reflection dataset"
}

# Connect to SQLite
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Open HDF5 and loop through patterns
with h5py.File(HDF5_FILE, "r") as f:
    for sg_key in f["patterns"]:
        sg_group = f["patterns"][sg_key]
        space_group = int(sg_key.split("_")[1])

        for pattern_key in sg_group:
            pattern_group = sg_group[pattern_key]
            
            # Generate pattern UUID
            pattern_id = str(uuid.uuid4())

            # Read intensity and twotheta
            intensity = np.array(pattern_group["intensity"])
            twotheta = np.array(pattern_group["twotheta"])

            # Serialize to binary blobs
            intensity_blob = intensity.tobytes()
            twotheta_blob = twotheta.tobytes()

            # Insert into the DB
            cursor.execute("""
                INSERT INTO Patterns2 (
                    pattern_id,
                    structure_id,
                    space_group,
                    radiation_type,
                    wavelength,
                    twotheta_min,
                    twotheta_max,
                    step_size,
                    resolution_u,
                    resolution_v,
                    resolution_w,
                    background_type,
                    generator_method,
                    generator_commit,
                    normalization_type,
                    axis_type,
                    pattern_format,
                    intensity_blob,
                    twotheta_blob,
                    has_structure_link,
                    comments
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                pattern_id,
                None,  # No structure_id link
                space_group,
                STATIC_METADATA["radiation_type"],
                STATIC_METADATA["wavelength"],
                STATIC_METADATA["twotheta_min"],
                STATIC_METADATA["twotheta_max"],
                STATIC_METADATA["step_size"],
                STATIC_METADATA["resolution_u"],
                STATIC_METADATA["resolution_v"],
                STATIC_METADATA["resolution_w"],
                STATIC_METADATA["background_type"],
                STATIC_METADATA["generator_method"],
                STATIC_METADATA["generator_commit"],
                STATIC_METADATA["normalization_type"],
                STATIC_METADATA["axis_type"],
                STATIC_METADATA["pattern_format"],
                intensity_blob,
                twotheta_blob,
                STATIC_METADATA["has_structure_link"],
                STATIC_METADATA["comments"]
            ))

# Finalize
conn.commit()
conn.close()
print("✅ All reflection patterns loaded into the Patterns table.")


✅ All reflection patterns loaded into the Patterns table.


### How to Query and Decode Later
Decoding is just the process of turning the binary blobs (BLOBs) back into usable NumPy arrays that you can plot, analyze, or feed into ML models.

In [19]:
import numpy as np
# Connect to SQLite
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

cursor.execute("SELECT intensity_blob, twotheta_blob FROM Patterns2 LIMIT 1")
row = cursor.fetchone()

intensity_array = np.frombuffer(row[0], dtype=np.float64)
twotheta_array = np.frombuffer(row[1], dtype=np.float64)

# Finalize
conn.commit()
conn.close()

### Verify table creation

query SQLite system

In [3]:
#list all tables in the database

conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in crystal_xrd.db:")
for table in tables:
    print(table[0])

#conn.close()


Tables in crystal_xrd.db:
Structures
Patterns


In [4]:
#check table Structures table schema
cursor.execute("PRAGMA table_info(Structures);")
structure_schema = cursor.fetchall()

print("Structures table schema:")
for col in structure_schema:
    print(col)


Structures table schema:
(0, 'structure_id', 'TEXT', 0, None, 1)
(1, 'source', 'TEXT', 0, None, 0)
(2, 'source_id', 'TEXT', 0, None, 0)
(3, 'generation_commit', 'TEXT', 0, None, 0)
(4, 'extinction_class', 'TEXT', 0, None, 0)
(5, 'composition', 'TEXT', 0, None, 0)
(6, 'lattice_a', 'REAL', 0, None, 0)
(7, 'lattice_b', 'REAL', 0, None, 0)
(8, 'lattice_c', 'REAL', 0, None, 0)
(9, 'alpha', 'REAL', 0, None, 0)
(10, 'beta', 'REAL', 0, None, 0)
(11, 'gamma', 'REAL', 0, None, 0)
(12, 'wyckoff_positions', 'TEXT', 0, None, 0)
(13, 'atom_sites_json', 'TEXT', 0, None, 0)
(14, 'creation_timestamp', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)


In [10]:
#check table Patterns table schema
cursor.execute("PRAGMA table_info(Patterns);")
structure_schema = cursor.fetchall()

print("Patterns table schema:")
for col in structure_schema:
    print(col)


Patterns table schema:
(0, 'pattern_id', 'TEXT', 0, None, 1)
(1, 'structure_id', 'TEXT', 0, None, 0)
(2, 'radiation_type', 'TEXT', 0, None, 0)
(3, 'wavelength', 'REAL', 0, None, 0)
(4, 'twotheta_min', 'REAL', 0, None, 0)
(5, 'twotheta_max', 'REAL', 0, None, 0)
(6, 'step_size', 'REAL', 0, None, 0)
(7, 'resolution_u', 'REAL', 0, None, 0)
(8, 'resolution_v', 'REAL', 0, None, 0)
(9, 'resolution_w', 'REAL', 0, None, 0)
(10, 'background_type', 'TEXT', 0, None, 0)
(11, 'generator_method', 'TEXT', 0, None, 0)
(12, 'generator_commit', 'TEXT', 0, None, 0)
(13, 'normalization_type', 'TEXT', 0, None, 0)
(14, 'axis_type', 'TEXT', 0, None, 0)
(15, 'pattern_format', 'TEXT', 0, None, 0)
(16, 'intensity_file', 'TEXT', 0, None, 0)
(17, 'has_structure_link', 'BOOLEAN', 0, '1', 0)
(18, 'comments', 'TEXT', 0, None, 0)
(19, 'generation_timestamp', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)


In [6]:
conn.close()

### Dropping Tables

In [7]:
## dropping a table
import sqlite3

# Connect to your database
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

# Drop the Structures table if it exists
cursor.execute("DROP TABLE IF EXISTS Structures;")
print("Dropped 'Structures' table.")

conn.commit()
conn.close()


Dropped 'Structures' table.


### Dropping columns